> 在线讲义：[第 51 章 · 后训练的数学：SFT = 最大似然、Bradley-Terry、KL 正则的闭式最优策略、DPO、GRPO](https://wikieden.github.io/math-and-eval-stats/lessons/51-post-training.html) · 本书《数学与实验统计》

# 第 51 章 · 后训练的数学：SFT = 最大似然、Bradley-Terry、KL 正则的闭式最优策略、DPO、GRPO

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/51-post-training.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 SFT | CE = KL + H | `sft_decomposition(data, PREF)` | 0.9380 = 0.0401 + 0.8979 |
| 第二步 BT | σ(Δr) | `bt_prob(1, 0)` | 0.7311 |
| 第三步 最优策略 | π_ref·e^{r/β} | `optimal_policy(1.0)` | [0.36, 0.5871, 0.053] |
| 第四步 DPO | 隐含奖励、损失 | `implied_reward`、`dpo_loss` | 差为常数；0.6931 → 0.1269 |
| 第五步 GRPO | 组内归一化、k3 | `grpo_adv(rw)`、`k3(r)` | ±1；0.1931、0、0.3069 |

例子：一个只有 3 个回答（A、B、C）的问题；参考策略 π_ref = [0.5, 0.3, 0.2]，奖励 r = [1, 2, 0]。
SFT 的交叉熵 = KL + 熵；Bradley-Terry 偏好概率；KL 正则目标的闭式最优策略；DPO 隐含奖励；GRPO 的组内归一化和 k3 KL 估计。
里程碑 ME-M31 的“5 选项玩具偏好数据上实现 DPO 和 GRPO 的一步更新”留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
PREF = np.array([0.5, 0.3, 0.2])
R = np.array([1.0, 2.0, 0.0])

**函数 `sft_decomposition`**：交叉熵 H(data, π) = KL(data ‖ π) + H(data)：最小化交叉熵 = 最小化 KL = 最大似然。返回 (交叉熵, KL, 熵)。

In [ ]:
def sft_decomposition(data: np.ndarray, pi: np.ndarray) -> tuple[float, float, float]:
    """交叉熵 H(data, π) = KL(data ‖ π) + H(data)：最小化交叉熵 = 最小化 KL = 最大似然。返回 (交叉熵, KL, 熵)。"""
    ce = -(data * np.log(pi)).sum()
    h = -(data * np.log(data)).sum()
    return float(ce), float(ce - h), float(h)

**函数 `bt_prob`**：Bradley-Terry：P(a ≻ b) = σ(r_a − r_b)。

In [ ]:
def bt_prob(ra: float, rb: float) -> float:
    """Bradley-Terry：P(a ≻ b) = σ(r_a − r_b)。"""
    return float(1 / (1 + np.exp(-(ra - rb))))

**函数 `optimal_policy`**：max E_π[r] − β·KL(π ‖ π_ref) 的解：π*(y) ∝ π_ref(y)·exp(r(y)/β)。

In [ ]:
def optimal_policy(beta: float, pref: np.ndarray = PREF, r: np.ndarray = R) -> np.ndarray:
    """max E_π[r] − β·KL(π ‖ π_ref) 的解：π*(y) ∝ π_ref(y)·exp(r(y)/β)。"""
    w = pref * np.exp(r / beta)
    return w / w.sum()

**函数 `objective`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def objective(pi: np.ndarray, beta: float) -> float:
    return float((pi * R).sum() - beta * (pi * np.log(pi / PREF)).sum())

**函数 `implied_reward`**：DPO 的关键反解：r(y) = β log(π*(y)/π_ref(y)) + β log Z。log Z 对所有回答相同，比较两个回答时消掉。

In [ ]:
def implied_reward(pi: np.ndarray, beta: float) -> np.ndarray:
    """DPO 的关键反解：r(y) = β log(π*(y)/π_ref(y)) + β log Z。log Z 对所有回答相同，比较两个回答时消掉。"""
    return beta * np.log(pi / PREF)

**函数 `dpo_loss`**：一对偏好 (y_w ≻ y_l) 的 DPO 损失 −log σ(β log(π(y_w)/π_ref(y_w)) − β log(π(y_l)/π_ref(y_l)))。

In [ ]:
def dpo_loss(pi: np.ndarray, w: int, l: int, beta: float) -> float:
    """一对偏好 (y_w ≻ y_l) 的 DPO 损失 −log σ(β log(π(y_w)/π_ref(y_w)) − β log(π(y_l)/π_ref(y_l)))。"""
    d = beta * (np.log(pi[w] / PREF[w]) - np.log(pi[l] / PREF[l]))
    return float(np.log1p(np.exp(-d)))

**函数 `grpo_adv`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def grpo_adv(rewards: np.ndarray) -> np.ndarray:
    return (rewards - rewards.mean()) / (rewards.std() + 1e-8)

**函数 `k3`**：GRPO 用的 KL 估计 k3 = r − log r − 1，r = π_ref/π_θ（对 π_θ 采样时无偏、恒非负）。

In [ ]:
def k3(ratio_ref_over_pi: np.ndarray) -> np.ndarray:
    """GRPO 用的 KL 估计 k3 = r − log r − 1，r = π_ref/π_θ（对 π_θ 采样时无偏、恒非负）。"""
    r = ratio_ref_over_pi
    return r - np.log(r) - 1

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 SFT = 最大似然 = 最小化 KL")
data = np.array([0.6, 0.3, 0.1])
for name, pi in (("π_ref", PREF), ("和数据一样", data)):
    ce, kl, h = sft_decomposition(data, pi)
    print(f"  数据分布 [0.6, 0.3, 0.1]，模型 {name}：交叉熵 {ce:.4f} = KL {kl:.4f} + 熵 {h:.4f}")

In [ ]:
print("\n第二步 Bradley-Terry 奖励模型")
for d in (0.0, 1.0, 3.0):
    print(f"  r_a − r_b = {d}：P(a ≻ b) = {bt_prob(d, 0):.4f}，奖励模型损失 −log σ = {-np.log(bt_prob(d, 0)):.4f}")

In [ ]:
print("\n第三步 KL 正则目标的闭式最优策略 π* ∝ π_ref·e^(r/β)")
for beta in (0.1, 1.0, 10.0):
    ps = optimal_policy(beta)
    print(f"  β = {beta}：π* = {np.round(ps, 4).tolist()}，目标值 {objective(ps, beta):.4f}（π_ref 时 {objective(PREF, beta):.4f}）")

In [ ]:
print("\n第四步 DPO：奖励可以用策略表示")
beta = 1.0
ps = optimal_policy(beta)
ir = implied_reward(ps, beta)
print(f"  β = 1：β log(π*/π_ref) = {np.round(ir, 4).tolist()}；减去 r = {R.tolist()} 后都等于 {np.round(ir - R, 4).tolist()}（= −log Z，共同的常数）")
print(f"  偏好对 B ≻ C：π = π_ref 时 DPO 损失 {dpo_loss(PREF, 1, 2, beta):.4f}，π = π* 时 {dpo_loss(ps, 1, 2, beta):.4f}")

In [ ]:
print("\n第五步 GRPO：组内归一化的优势和 k3 KL 估计")
rw = np.array([1.0, 0.0, 0.0, 1.0, 1.0, 0.0])
print(f"  6 个回答的奖励 {rw.tolist()} → 优势 {np.round(grpo_adv(rw), 4).tolist()}")
print(f"  k3(π_ref/π_θ) 在比率 0.5、1、2 时 = {np.round(k3(np.array([0.5, 1.0, 2.0])), 4).tolist()}（恒 ≥ 0）")

## 3. 代码里才看得到的

1. **测试里验证 π* 真的是最优**：随机抽 200 个分布，目标值都不超过 π* 的。
2. **−log σ(d) 写成 log(1 + e^{−d})**：用 `log1p` 在 d 大时更准（第 22 章）。
3. **没有做梯度更新**：在 5 选项的玩具偏好数据上实现 DPO 和 GRPO 的一步更新留给自己做（里程碑 ME-M31）。

## 4. 自己改一改

1. 把 π_ref 改成 [0.1, 0.1, 0.8]（参考策略很偏向奖励最低的 C），β = 1 时 π* 是多少？
2. 在 `dpo_loss` 里把 β 改成 0.1，同样的 π* 损失变成多少？说明 β 在 DPO 里起什么作用。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> SFT 交叉熵等于 KL 加常数，只能模仿；
> BT 用分数差的 sigmoid，奖励模型是逻辑回归；
> 奖励减 β 倍 KL，最优是参考乘 e 的 r 除 β；
> 反解奖励 β log 比值，常数在差里消掉，就是 DPO；
> GRPO 组内减均值除标准差，KL 用 k3。